# Grid Objects in Seaborn

Seaborn's grid classes give you fine-grained control over multi-panel figures. While the figure-level functions (`relplot`, `catplot`, `displot`) create grids automatically, the underlying grid objects — `FacetGrid`, `PairGrid`, and `JointGrid` — let you build custom layouts and map arbitrary plotting functions.

This notebook covers:
- `sns.FacetGrid` — faceted grids with `map()` and `map_dataframe()`
- `sns.PairGrid` — custom pairwise plot grids
- `sns.JointGrid` — custom bivariate plots with marginals
- Customizing titles, axes, and legends

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline
sns.set_theme(style='ticks')

In [ ]:
iris = sns.load_dataset('iris')
penguins = sns.load_dataset('penguins').dropna()

print(f'iris: {iris.shape}')
print(f'penguins: {penguins.shape}')
iris.head()

---
## FacetGrid

`FacetGrid` creates a matrix of panels defined by `row` and `col` variables. You then **map** a plotting function to each panel. This is the engine behind `relplot()`, `catplot()`, and `displot()`.

In [ ]:
# Basic FacetGrid: histogram of sepal_length for each species
g = sns.FacetGrid(iris, col='species', height=4, aspect=1)
g.map(plt.hist, 'sepal_length', bins=15, color='steelblue', edgecolor='white')
g.set_titles('{col_name}')
g.fig.suptitle('Sepal Length Histograms by Species', y=1.03)
plt.show()

### `map()` vs `map_dataframe()`

- `map(func, col_name, ...)` passes arrays: suitable for matplotlib functions like `plt.hist`, `plt.scatter`.
- `map_dataframe(func, x=..., y=..., ...)` passes a DataFrame subset: suitable for seaborn functions that expect `data=`.

In [ ]:
# Using map with plt.scatter
g = sns.FacetGrid(iris, col='species', height=4)
g.map(plt.scatter, 'sepal_length', 'sepal_width', alpha=0.7, s=50)
g.set_titles('{col_name}')
g.set_axis_labels('Sepal Length (cm)', 'Sepal Width (cm)')
plt.show()

In [ ]:
# Using map_dataframe with a seaborn function
g = sns.FacetGrid(penguins, col='island', height=4)
g.map_dataframe(
    sns.scatterplot,
    x='bill_length_mm', y='bill_depth_mm',
    hue='species', style='species',
)
g.set_titles('{col_name}')
g.add_legend()
plt.show()

In [ ]:
# FacetGrid with both row and col
g = sns.FacetGrid(
    penguins, row='sex', col='species',
    height=3, aspect=1.1, margin_titles=True,
)
g.map_dataframe(sns.histplot, x='body_mass_g', bins=15, color='teal')
g.set_titles(row_template='{row_name}', col_template='{col_name}')
g.fig.suptitle('Body Mass by Species and Sex', y=1.02)
plt.show()

### Customizing FacetGrid Appearance

In [ ]:
g = sns.FacetGrid(
    penguins, col='species',
    height=4, aspect=1,
    sharex=True, sharey=True,
)
g.map_dataframe(sns.kdeplot, x='flipper_length_mm', fill=True)

# Add vertical reference lines for each panel
for ax in g.axes.flat:
    ax.axvline(x=penguins['flipper_length_mm'].mean(), color='red',
               linestyle='--', linewidth=1, label='Overall Mean')

g.set_titles('{col_name}')
g.set_axis_labels('Flipper Length (mm)', 'Density')
g.fig.suptitle('Flipper Length KDE with Overall Mean', y=1.03)
plt.show()

---
## PairGrid

`PairGrid` creates a matrix of axes where each cell shows the relationship between a pair of variables. Unlike `pairplot()` (which is a convenience wrapper), `PairGrid` lets you map **different functions** to the diagonal, upper triangle, and lower triangle.

In [ ]:
# Default PairGrid with uniform mapping
g = sns.PairGrid(iris, hue='species', height=2.5)
g.map(sns.scatterplot, alpha=0.6)
g.add_legend()
plt.show()

In [ ]:
# Different functions on diagonal vs off-diagonal
g = sns.PairGrid(iris, hue='species', height=2.5)
g.map_diag(sns.histplot, kde=True)
g.map_offdiag(sns.scatterplot, alpha=0.5)
g.add_legend()
g.fig.suptitle('Iris: Histograms on Diagonal, Scatter Off-Diagonal', y=1.02)
plt.show()

In [ ]:
# Different functions for upper and lower triangles
g = sns.PairGrid(penguins, hue='species', height=2.5,
                 vars=['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm'])
g.map_upper(sns.scatterplot, alpha=0.5, s=20)
g.map_lower(sns.kdeplot, levels=4, fill=True, alpha=0.4)
g.map_diag(sns.kdeplot, fill=True, alpha=0.5)
g.add_legend()
g.fig.suptitle('PairGrid: Scatter (upper), KDE (lower & diagonal)', y=1.02)
plt.show()

In [ ]:
# Selecting a subset of variables
g = sns.PairGrid(
    iris,
    x_vars=['sepal_length', 'sepal_width'],
    y_vars=['petal_length', 'petal_width'],
    hue='species', height=3.5,
)
g.map(sns.scatterplot, alpha=0.6)
g.add_legend()
g.fig.suptitle('Sepal vs Petal Measurements', y=1.02)
plt.show()

---
## JointGrid

`JointGrid` creates a central axes for a bivariate plot with marginal axes on the top and right for univariate distributions. `jointplot()` is a convenience wrapper; `JointGrid` gives you full control.

In [ ]:
# Basic JointGrid
g = sns.JointGrid(
    data=penguins,
    x='flipper_length_mm', y='body_mass_g',
    height=6,
)
g.plot(sns.scatterplot, sns.histplot)
plt.show()

In [ ]:
# Custom JointGrid: scatter + KDE + rug on marginals
g = sns.JointGrid(
    data=penguins,
    x='bill_length_mm', y='bill_depth_mm',
    hue='species', height=7,
)
g.plot_joint(sns.scatterplot, alpha=0.6)
g.plot_marginals(sns.kdeplot, fill=True, alpha=0.4)
g.fig.suptitle('Bill Dimensions with Marginal KDEs', y=1.02)
plt.show()

In [ ]:
# JointGrid with KDE contours in the center
g = sns.JointGrid(
    data=iris, x='sepal_length', y='sepal_width',
    height=6,
)
g.plot_joint(sns.kdeplot, fill=True, cmap='Blues', levels=8)
g.plot_joint(sns.scatterplot, color='navy', s=15, alpha=0.3)
g.plot_marginals(sns.histplot, kde=True, color='steelblue')
plt.show()

In [ ]:
# Adding a reference line to the joint plot
g = sns.JointGrid(
    data=penguins,
    x='flipper_length_mm', y='body_mass_g',
    height=6,
)
g.plot_joint(sns.scatterplot, hue=penguins['species'], alpha=0.6)
g.plot_marginals(sns.histplot, kde=True)

# Add a regression line manually
from numpy.polynomial.polynomial import polyfit
x = penguins['flipper_length_mm']
y = penguins['body_mass_g']
b, m = polyfit(x, y, 1)
x_range = np.linspace(x.min(), x.max(), 100)
g.ax_joint.plot(x_range, b + m * x_range, 'r--', linewidth=2, label='Linear Fit')
g.ax_joint.legend()
plt.show()

## Customizing Grid Titles and Axes

In [ ]:
g = sns.FacetGrid(
    penguins, col='species', hue='sex',
    height=4, aspect=1, palette='Set1',
)
g.map_dataframe(sns.scatterplot, x='bill_length_mm', y='bill_depth_mm', alpha=0.7)

# Customize titles
g.set_titles(col_template='Species: {col_name}', size=13)

# Customize axis labels
g.set_axis_labels('Bill Length (mm)', 'Bill Depth (mm)', fontsize=11)

# Add legend with custom title
g.add_legend(title='Sex')

# Despine for cleaner look
sns.despine()

g.fig.suptitle('Bill Dimensions by Species and Sex', fontsize=14, y=1.04)
plt.show()

## Key Takeaways

| Grid Class | Purpose | Convenience Wrapper |
|---|---|---|
| `FacetGrid` | Small multiples across categories | `relplot()`, `catplot()`, `displot()` |
| `PairGrid` | Custom pairwise variable plots | `pairplot()` |
| `JointGrid` | Bivariate plot with marginal distributions | `jointplot()` |

- Use **convenience wrappers** for quick exploration and **grid classes** when you need different plot types per region or manual customization.
- `map()` passes numpy arrays; `map_dataframe()` passes a filtered DataFrame — choose based on whether your plotting function expects arrays or `data=`.
- `PairGrid` with `map_upper`, `map_lower`, and `map_diag` is the standard pattern for rich pairwise visualizations.
- `JointGrid` gives you separate handles for the joint and marginal axes via `g.ax_joint` and `g.ax_marg_x` / `g.ax_marg_y`.